# Kaggle GPU — Fine-tune VietOCR on this competition's data

Teaches a **lightweight** Vietnamese recognizer the exact spelling/diacritics of the
GT labels. Pipeline:
1. Detect text lines (EasyOCR/CRAFT) on train images, crop them.
2. Recognize each crop with base VietOCR, **align** the prediction to the matching
   span of the GT `ocr_text` (correct diacritics) -> `(crop, label)` pair.
3. **Fine-tune** VietOCR on those pairs.
4. Run the fine-tuned model on test images -> `ocr_vietocr_ft_test.parquet`.

## Setup
1. New Notebook **inside the competition** (data auto-mounts).
2. Sidebar: **GPU T4 x2**, **Internet ON**.
3. Run **Cell 1**, then **Run > Restart & clear cell outputs** (Pillow pin), then **Run All**.
4. Long run (~2-4h). Use **Save Version > Save & Run All (Commit)** for background.
5. Download `ocr_vietocr_ft_test.parquet` (and `_all`) into local `cache/`.

> Quick first run: set `MAX_TRAIN_IMAGES=1500` (Cell 3), `ITERS=3000` (Cell 4).
> Then eyeball the sample labels printed at the end of Cell 3 before committing.

In [ ]:
# Pillow pin AFTER installing OCR deps (else ImageFont/_util.is_directory ImportError).
# >>> After this cell: RESTART kernel, then Run All. <<<
!pip install -q easyocr vietocr rapidfuzz
!pip install -q --force-reinstall 'Pillow==10.4.0'
import PIL, torch
print('Pillow', PIL.__version__, '| CUDA', torch.cuda.is_available(),
      '|', (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'))

In [ ]:
import os, re, zipfile, unicodedata, random
from pathlib import Path
import numpy as np, pandas as pd, cv2
from PIL import Image
from tqdm.auto import tqdm

KIN = Path('/kaggle/input'); ROOT = None
for p in KIN.rglob('test.csv'):
    ROOT = p.parent; break
assert ROOT is not None, 'competition data not mounted'
WORK = Path('/kaggle/working'); print('ROOT =', ROOT)

def _ensure_dir(cands, zips, out):
    for rel in cands:
        d = ROOT/rel
        if d.is_dir() and any(d.glob('*.jpg')): return d
    for zp in zips:
        z = ROOT/zp
        if z.is_file():
            ex = WORK/out
            if not (ex.exists() and any(ex.rglob('*.jpg'))):
                ex.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(z) as zf: zf.extractall(ex)
            for d in [ex,*[c for c in ex.rglob('*') if c.is_dir()]]:
                if any(d.glob('*.jpg')): return d
    raise FileNotFoundError(out)

TRAIN_DIR = _ensure_dir(['train_images/train_images','train_images'], ['train_images.zip'], 'train_images')
TEST_DIR  = _ensure_dir(['test_images/images','images','test_images'], ['images.zip','test_images.zip'], 'test_images')
labels = pd.read_csv(ROOT/'train_labels.csv', keep_default_na=False)
labels['ocr_text'] = labels['ocr_text'].astype(str).str.strip()
test_ids = pd.read_csv(ROOT/'test.csv')['image_id'].tolist()
print('train imgs', len(list(TRAIN_DIR.glob('*.jpg'))), '| test imgs', len(list(TEST_DIR.glob('*.jpg'))), '| labeled', len(labels))

# diacritic-fold WITHOUT unidecode: NFD strips Vietnamese tone marks (combining 'Mn'); handle d-stroke.
def fold(s):
    s = unicodedata.normalize('NFD', str(s).lower())
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = s.replace(chr(0x111), 'd')
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def clean_ocr(t, maxlen=500):
    t = unicodedata.normalize('NFC', str(t)).replace(chr(10),' ').replace(chr(9),' ')
    t = re.sub(r'\s+', ' ', t).strip()
    toks = t.split(); ded = [toks[0]] if toks else []
    for w in toks[1:]:
        if w.lower() != ded[-1].lower(): ded.append(w)
    t = ' '.join(ded); return t[:maxlen].rstrip() if len(t) > maxlen else t

In [ ]:
# ---- Build aligned line dataset: (line crop -> GT span with correct diacritics) ----
import easyocr
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg
from rapidfuzz import fuzz

MAX_TRAIN_IMAGES = None   # set e.g. 1500 for a faster first run; None = all
MATCH_MIN = 75            # min fuzzy score (0-100) to accept a crop's GT-aligned label

detector = easyocr.Reader(['vi','en'], gpu=True, verbose=False)
base_cfg = Cfg.load_config_from_name('vgg_transformer')
base_cfg['device'] = 'cuda:0'; base_cfg['predictor']['beamsearch'] = False
base_rec = Predictor(base_cfg)
VOCAB = set(base_cfg['vocab'])
LINES = WORK/'lines'; LINES.mkdir(exist_ok=True)

def best_gt_span(pred, gt_tokens, max_span=10):
    pf = fold(pred)
    if len(pf) < 2: return None, 0
    best, bs, n = None, 0, len(gt_tokens)
    for i in range(n):
        for L in range(1, max_span+1):
            if i+L > n: break
            span = ' '.join(gt_tokens[i:i+L])
            sc = fuzz.ratio(pf, fold(span))
            if sc > bs: bs, best = sc, span
        if bs == 100: break
    return best, bs

rows = labels[labels.ocr_text != ''].copy()
if MAX_TRAIN_IMAGES: rows = rows.sample(min(MAX_TRAIN_IMAGES, len(rows)), random_state=42)

pairs = []; idx = 0
for _, r in tqdm(rows.iterrows(), total=len(rows), desc='align'):
    img = cv2.imread(str(TRAIN_DIR/(r.image_id + '.jpg')))
    if img is None: continue
    gt_tokens = r.ocr_text.split()
    try:
        horiz,_ = detector.detect(img, min_size=15, text_threshold=0.6); boxes = horiz[0] if horiz else []
    except Exception:
        boxes = []
    for b in boxes:
        x0,x1,y0,y1 = [int(v) for v in b]
        x0,y0 = max(0,x0),max(0,y0); x1,y1 = min(img.shape[1],x1),min(img.shape[0],y1)
        if x1-x0 < 6 or y1-y0 < 6: continue
        crop = img[y0:y1, x0:x1]
        try:
            pred = base_rec.predict(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)))
        except Exception:
            continue
        span, sc = best_gt_span(pred, gt_tokens)
        if span is None or sc < MATCH_MIN: continue
        label = ''.join(ch for ch in span if ch in VOCAB).strip()
        if len(label) < 1: continue
        fn = 'l%06d.png' % idx; cv2.imwrite(str(LINES/fn), crop); idx += 1
        pairs.append((fn, label))

print('\nbuilt', len(pairs), 'aligned line pairs from', len(rows), 'images')
random.seed(0); random.shuffle(pairs)
nval = max(200, int(0.03*len(pairs)))
val_pairs, train_pairs = pairs[:nval], pairs[nval:]
with open(LINES/'train_ann.txt','w',encoding='utf-8') as f:
    for fn,lb in train_pairs: f.write(fn + '\t' + lb + '\n')
with open(LINES/'val_ann.txt','w',encoding='utf-8') as f:
    for fn,lb in val_pairs: f.write(fn + '\t' + lb + '\n')
print('train', len(train_pairs), '| val', len(val_pairs))
print('SAMPLE LABELS (eyeball these — should be clean Vietnamese):')
for _,lb in train_pairs[:12]: print('  ', lb)

In [ ]:
# ---- Fine-tune VietOCR ----
import numpy as np, time
# Shim NumPy 2.0 removals (vietocr + imgaug are numpy-1.x era code)
if not hasattr(np, 'sctypes'):
    np.sctypes = {'int':[np.int8,np.int16,np.int32,np.int64],
                  'uint':[np.uint8,np.uint16,np.uint32,np.uint64],
                  'float':[np.float16,np.float32,np.float64],
                  'complex':[np.complex64,np.complex128],
                  'others':[bool,object,bytes,str,np.void]}
for _n,_t in [('bool',bool),('float',float),('int',int),('object',object),('str',str),('complex',complex)]:
    if not hasattr(np,_n): setattr(np,_n,_t)
_np_fromstring = np.fromstring
def _fromstring_compat(string, dtype=float, count=-1, sep=''):
    return np.frombuffer(string, dtype=dtype, count=count) if sep=='' else _np_fromstring(string, dtype=dtype, count=count, sep=sep)
np.fromstring = _fromstring_compat

from vietocr.model.trainer import Trainer
from vietocr.tool.config import Cfg

ITERS = 15000   # lower (e.g. 3000) for a quick first run
cfg = Cfg.load_config_from_name('vgg_transformer')
cfg['device'] = 'cuda:0'
cfg['dataloader'] = {'num_workers': 2}
cfg['dataset'] = {'name': 'ura_%d' % int(time.time()), 'data_root': str(LINES),
                  'train_annotation':'train_ann.txt', 'valid_annotation': None,   # skip buggy internal val
                  'image_height':32, 'image_min_width':32, 'image_max_width':512}
cfg['trainer'] = {'batch_size':32, 'print_every':200, 'valid_every':10**9, 'iters':ITERS,
                  'export':'./vietocr_ft.pth', 'checkpoint':'./ckpt.pth',
                  'log':'./train.log', 'metrics':5000}
trainer = Trainer(cfg, pretrained=True)
trainer.train()
trainer.save_weights('./vietocr_ft.pth')   # manual save (no val-triggered save)
print('fine-tuned model saved -> ./vietocr_ft.pth')

In [ ]:
# ---- Inference with fine-tuned model -> parquet (same schema as run_ocr.py) ----
ft_cfg = Cfg.load_config_from_name('vgg_transformer')
ft_cfg['device'] = 'cuda:0'; ft_cfg['predictor']['beamsearch'] = False
ft_cfg['weights'] = './vietocr_ft.pth'
ft_rec = Predictor(ft_cfg)

def ocr_image(path):
    img = cv2.imread(str(path))
    if img is None: return {'ocr_text':'','raw_text':'','mean_conf':0.0,'n_boxes':0,'n_chars':0}
    try:
        horiz,_ = detector.detect(img, min_size=15, text_threshold=0.6); boxes = horiz[0] if horiz else []
    except Exception:
        boxes = []
    items = []
    for b in boxes:
        x0,x1,y0,y1 = [int(v) for v in b]
        x0,y0 = max(0,x0),max(0,y0); x1,y1 = min(img.shape[1],x1),min(img.shape[0],y1)
        if x1-x0 < 6 or y1-y0 < 6: continue
        items.append((y0, x0, Image.fromarray(cv2.cvtColor(img[y0:y1,x0:x1], cv2.COLOR_BGR2RGB))))
    if not items: return {'ocr_text':'','raw_text':'','mean_conf':0.0,'n_boxes':0,'n_chars':0}
    try: texts = ft_rec.predict_batch([it[2] for it in items])
    except Exception: texts = [ft_rec.predict(it[2]) for it in items]
    ys = [it[0] for it in items]; band = max(8.0, (max(ys)-min(ys))/40.0)
    order = sorted(range(len(items)), key=lambda i:(round(items[i][0]/band), items[i][1]))
    text = clean_ocr(' '.join(texts[i] for i in order if str(texts[i]).strip()))
    return {'ocr_text':text,'raw_text':text,'mean_conf':0.0,'n_boxes':len(items),'n_chars':len(text)}

def run(ids, img_dir, out, save_every=200):
    done = {}
    if Path(out).exists():
        done = {r['image_id']:r for r in pd.read_parquet(out).to_dict('records')}
    recs = list(done.values()); pend = [i for i in ids if i not in done]
    print(out, 'pending', len(pend), '/', len(ids))
    for k,iid in enumerate(tqdm(pend)):
        r = ocr_image(img_dir/(iid + '.jpg')); r['image_id'] = iid; recs.append(r)
        if (k+1) % save_every == 0: pd.DataFrame(recs).to_parquet(out, index=False)
    pd.DataFrame(recs).to_parquet(out, index=False)
    print('saved', out, '| fill', (pd.DataFrame(recs).ocr_text.str.strip() != '').mean())

run(test_ids, TEST_DIR, '/kaggle/working/ocr_vietocr_ft_test.parquet')
run(labels['image_id'].tolist(), TRAIN_DIR, '/kaggle/working/ocr_vietocr_ft_all.parquet')
print('\nDONE — download ocr_vietocr_ft_test.parquet (+ _all) from Output.')